In [1]:
# Import packages
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('data/multimodalwithres/Multimodal.csv')
df

,LOC_ID,CUSTOMER_ID,TX_ID,TX_DATE,TX_TME,ITEM_ID,SUBGROUP_ID,NET_SALES_UNITS,NET_SALES_AMT
0,8,39260,748538,2003/3/19,153300,6100,459,0.949816,3.854712
1,8,30743,876237,2003/3/19,201000,1746,974,1.180286,1.834784
2,8,30743,876237,2003/3/19,201000,1746,974,1.092798,1.699623
3,8,28346,746752,2003/3/19,161400,1315,795,1.039463,2.955283
4,8,77899,925250,2003/3/19,154100,8854,699,0.862407,3.591753
...,...,...,...,...,...,...,...,...,...
790422,8,88580,797489,2/20/2020,121000,6240,109,0.868706,1.691392
790423,8,20238,658395,2/20/2020,121000,8102,722,1.037933,2.790068
790424,8,20238,658395,2/20/2020,121000,8102,722,1.090351,3.543588
790425,8,20238,658395,2/20/2020,121000,4533,434,0.910742,1.599769


In [3]:
# The date period of the data
# Convert 'TX_DATE' column to datetime format
def parse_date(date_str):
    for fmt in ('%m/%d/%Y', '%m/%d/%y'):  # List formats to try
        try:
            return pd.to_datetime(date_str, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df['TX_DATE'] = df['TX_DATE'].apply(parse_date)

# Find first and last dates of data
first_date = df['TX_DATE'].min()
last_date = df['TX_DATE'].max()

print(f"First date in the dataset: {first_date}")
print(f"Last date in the dataset: {last_date}")

First date in the dataset: 2019-03-13 00:00:00
Last date in the dataset: 2020-02-21 00:00:00


In [4]:
# Check columns
print(df.columns)

# Since ITEM_ID exists, factorize
df['item_no'], item_labels = pd.factorize(df['ITEM_ID'])

# Create item mapping table if needed
item_mapping = pd.DataFrame({
    'ITEM_ID': item_labels,
    'item_no': range(len(item_labels))
})

df

Index(['LOC_ID', 'CUSTOMER_ID', 'TX_ID', 'TX_DATE', 'TX_TME', 'ITEM_ID',
       'SUBGROUP_ID', 'NET_SALES_UNITS', 'NET_SALES_AMT'],
      dtype='object')


,LOC_ID,CUSTOMER_ID,TX_ID,TX_DATE,TX_TME,ITEM_ID,SUBGROUP_ID,NET_SALES_UNITS,NET_SALES_AMT,item_no
0,8,39260,748538,NaT,153300,6100,459,0.949816,3.854712,0
1,8,30743,876237,NaT,201000,1746,974,1.180286,1.834784,1
2,8,30743,876237,NaT,201000,1746,974,1.092798,1.699623,1
3,8,28346,746752,NaT,161400,1315,795,1.039463,2.955283,2
4,8,77899,925250,NaT,154100,8854,699,0.862407,3.591753,3
...,...,...,...,...,...,...,...,...,...,...
790422,8,88580,797489,2020-02-20,121000,6240,109,0.868706,1.691392,807
790423,8,20238,658395,2020-02-20,121000,8102,722,1.037933,2.790068,113
790424,8,20238,658395,2020-02-20,121000,8102,722,1.090351,3.543588,113
790425,8,20238,658395,2020-02-20,121000,4533,434,0.910742,1.599769,916


In [9]:
# Step 1: Calculate price for each transaction
df['itemPrice'] = df['NET_SALES_AMT'] / df['NET_SALES_UNITS']

# Step 2: Group by item_no to calculate average price
item_price_mapping = df.groupby('item_no')['itemPrice'].mean().reset_index()

# Step 3: Round the price if needed (optional)
item_price_mapping['itemPrice'] = item_price_mapping['itemPrice'].round(4)

In [10]:
print(item_price_mapping)

      item_no  itemPrice
0           0     4.7168
1           1     1.9086
2           2     2.6970
3           3     4.0740
4           4     2.7475
...       ...        ...
4995     4995     1.2266
4996     4996     5.0490
4997     4997    23.9975
4998     4998     4.0711
4999     4999     7.2034

[5000 rows x 2 columns]


In [11]:
item_price_mapping.to_csv('item_price_mapping.txt', sep='\t', index=False, header=True)
